# 网架统一的区域验证

只在 `Network` 中定义网架。`model.py` 共用 LP/SOCP 模型，`vertify.py` 共用独立 AC 和四方法实验，`plot.py` 共用绘图。

在下面修改 `CASE` 即可切换网架：`case33bw` 为固定设备的三节点可调度域截面；`four_bus_five_corridor` 为含建设方案的可规划域。固定网架不设置建设预算。

`RECOMPUTE=False` 明确读取归档实验，表中的耗时来自原求解记录；`True` 使用当前代码重新计算。每次实验只保存一个 `result.npz`，其中 AC 标签只存一次；FR、MR、差集颜色和表格都从该文件计算。

In [1]:
from importlib import import_module
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, IFrame
from vertify import BenchmarkResult, MethodBenchmark, validate_power_flow, METHODS, METHOD_NAMES
from plot import save_method_comparison

CASE = "case33bw"  # 可改为 "four_bus_five_corridor"。
network = import_module(f"Network.{CASE}").network
RECOMPUTE = False
BUDGETS = (20000., 40000., 60000., np.inf) if network.planning else (np.inf,)
DIVISIONS = 228 if network.planning else 192
RADIAL_TOLERANCE = 0.002
OUTPUT = Path("results")/network.name


## 模型与独立参考

三个独立坐标是 `network.load_nodes` 中节点的有功负荷；其余节点 P/Q 固定。负荷无功关系、线路参数、电压限值和源端限值全部由网架提供。

LP 忽略线路损耗；SOCP 保留功率平衡和完整电压降，将 $P_{ij}^2+Q_{ij}^2=v_i\ell_{ij}$ 放松为 $P_{ij}^2+Q_{ij}^2\le v_i\ell_{ij}$。混合方法先完成 LP 切割，再由同一个 SOCP 模型精修外围顶点。两种 SOCP 使用同一径向停止精度，固定背景负荷在认证内点生成过程中保持不变。

独立 AC 保留电流等式，通过另行实现的支路递推求解；未确定点交给显式非凸 AC 模型，仍未确定时不发布 FR/MR。AC 标签不参与生成割或停止判断。

33 节点算例采用原始 0.9–1.1 p.u. 电压限值；原始线路热限未启用，10 MVA 仅为标幺基准。

In [2]:
if not network.planning:
    validation = validate_power_flow(network)
    display(pd.DataFrame([{
        "最低电压 (p.u.)": validation["baseline"]["minimum_voltage_pu"],
        "最低电压节点": validation["baseline"]["minimum_voltage_bus"],
        "有功损耗 (kW)": validation["baseline"]["active_loss_kw"],
        "独立节点潮流最大电压差": validation["maximum_voltage_difference_pu"],
    }]))


,最低电压 (p.u.),最低电压节点,有功损耗 (kW),独立节点潮流最大电压差
0,0.91309,18,202.677126,7.438494e-15


## 求解与比较

规划实验逐预算比较建设方案并集；固定网架实验用 64³、128³、192³ 网格检查采样稳定性。网格只用于事后评价。

$$\mathrm{FR}=\frac{\mathrm{Vol}(\widehat{\mathcal D}\setminus\mathcal D_{AC})}{\mathrm{Vol}(\widehat{\mathcal D})}\times100\%,\qquad
\mathrm{MR}=\frac{\mathrm{Vol}(\mathcal D_{AC}\setminus\widehat{\mathcal D})}{\mathrm{Vol}(\mathcal D_{AC})}\times100\%.$$

总时间包含建模、求解、几何构域与网格归属判定，混合方法计入 LP 阶段。AC 扫描时间单列；公共网格创建、许可证启动和绘图导出不计。

In [3]:
grids = (DIVISIONS,) if network.planning else (64, 128, DIVISIONS)
runs = []
for divisions in grids:
    folder = OUTPUT if divisions == DIVISIONS else OUTPUT/f"grid_{divisions}"
    result = (MethodBenchmark(network, BUDGETS, divisions, RADIAL_TOLERANCE).run(folder)
              if RECOMPUTE else BenchmarkResult.load(folder))
    runs.append(result)

names = dict(zip(METHODS, METHOD_NAMES))
summary = pd.DataFrame(result.summary)
summary["method"] = summary["method"].map(names)
columns = ["budget", "method", "fr_percent", "mr_percent", "total_seconds"] if network.planning else [
    "method", "fr_percent", "mr_percent", "total_seconds"]
display(summary[columns].rename(columns={"budget":"预算 (元)", "method":"方法", "fr_percent":"FR (%)",
    "mr_percent":"MR (%)", "total_seconds":"总计算时间 (s)"}).style.format(precision=5))
if len(runs) > 1:
    refinement = pd.DataFrame([dict(divisions=r.metadata["divisions"], **row) for r in runs for row in r.summary])
    refinement["method"] = refinement["method"].map(names)
    display(refinement.pivot(index="divisions", columns="method", values="fr_percent").round(5))


,方法,FR (%),MR (%),总计算时间 (s)
0,纯 SOCP 切割,0.03298,0.00000,1.78580
1,线性 + SOCP 精修,0.03277,0.00000,1.83599
2,AC 数值参考,0.00000,0.00000,52.78613
3,纯线性切割,34.47973,0.00000,0.39693


method,AC 数值参考,纯 SOCP 切割,纯线性切割,线性 + SOCP 精修
divisions,,,,
64,0.0,0.04918,34.41033,0.04216
128,0.0,0.03183,34.48403,0.03411
192,0.0,0.03298,34.47973,0.03277


## 区域图

蓝色表示重合，**红色表示遗漏、黄色表示多余**。四幅图同步旋转，“仅差异”可查看薄误差层。所有图层使用同一份网格分类，不对非凸方案并集取整体凸包。

有限网格上的 MR＝0 仅表示未检出遗漏。数学上的 SOCP 内外域停止条件与网格估计分别检查。

In [4]:
page = save_method_comparison(result, OUTPUT)
display(IFrame(src=page.as_posix(), width="100%", height=1220))


代码和数据分工：

- `Network/`：原始网架、设备、固定负荷及规划方案生成。
- `model.py`：共用电气仿射方程、LP/SOCP 子问题和规划主问题。
- `vertify.py`：独立 AC、四方法实验和唯一结果文件；没有网架专属模型。
- `plot.py`：从结果对象绘图；HTML 是派生展示文件。
- `results/<网架名>/result.npz`：原始分类、配置、计时及去重后的几何记录。粗网格属于独立实验，放在对应 `grid_*` 目录。

既有实验已无损迁移，原时间和来源信息保留；没有把迁移耗时当成重新求解时间。